<p style="font-weight:bold; color:red; size: 100px;"> This is Anomaly Detection for our cloud Environment</p>

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")


Project root: c:\Users\Abhishek\Desktop\CloudSentinel-AI
Raw data directory: c:\Users\Abhishek\Desktop\CloudSentinel-AI\data\raw
Processed data directory: c:\Users\Abhishek\Desktop\CloudSentinel-AI\data\processed
Model directory: c:\Users\Abhishek\Desktop\CloudSentinel-AI\models


In [7]:
RAW_DATA_FILE = RAW_DATA_DIR / "cloud_resource_metrics.csv"

cloud_df = pd.read_csv(
    RAW_DATA_FILE,
    parse_dates=["timestamp"]
)

print("Dataset loaded successfully!")
print(f"Shape: {cloud_df.shape}")

Dataset loaded successfully!
Shape: (14400, 11)


In [8]:
cloud_df.head()

,timestamp,resource_id,resource_type,cpu_utilization,memory_utilization,network_in,request_count,storage_usage,estimated_cost,is_anomaly,anomaly_type
0,2026-08-01 00:00:00,EC2-001,Web Server,19.087236,24.906175,74.708176,899.351067,54.324431,0.059451,0,normal
1,2026-08-01 00:15:00,EC2-001,Web Server,21.372462,28.249815,53.193879,739.607109,56.138281,0.060514,0,normal
2,2026-08-01 00:30:00,EC2-001,Web Server,20.178249,26.371394,57.415241,773.460620,54.555520,0.059651,0,normal
3,2026-08-01 00:45:00,EC2-001,Web Server,16.712111,24.016472,77.400133,922.063367,53.726745,0.058352,0,normal
4,2026-08-01 01:00:00,EC2-001,Web Server,19.658934,25.695524,63.384328,790.523342,54.572166,0.059368,0,normal


In [9]:
print(cloud_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14400 entries, 0 to 14399
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   timestamp           14400 non-null  datetime64[ns]
 1   resource_id         14400 non-null  object        
 2   resource_type       14400 non-null  object        
 3   cpu_utilization     14400 non-null  float64       
 4   memory_utilization  14400 non-null  float64       
 5   network_in          14400 non-null  float64       
 6   request_count       14400 non-null  float64       
 7   storage_usage       14400 non-null  float64       
 8   estimated_cost      14400 non-null  float64       
 9   is_anomaly          14400 non-null  int64         
 10  anomaly_type        14400 non-null  object        
dtypes: datetime64[ns](1), float64(6), int64(1), object(3)
memory usage: 1.2+ MB
None


In [10]:
print("Anomaly distribution:")
print(cloud_df["is_anomaly"].value_counts())

print("\nAnomaly types:")
print(cloud_df["anomaly_type"].value_counts())

Anomaly distribution:
is_anomaly
0    14300
1      100
Name: count, dtype: int64

Anomaly types:
anomaly_type
normal            14300
cpu_spike            25
network_spike        20
memory_spike         20
cost_spike           20
combined_spike       15
Name: count, dtype: int64


In [11]:
feature_columns = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "storage_usage",
    "estimated_cost"
]

X = cloud_df[feature_columns].copy()

print("Features used for ML:")
print(feature_columns)

print("\nFeature matrix shape:")
print(X.shape)

Features used for ML:
['cpu_utilization', 'memory_utilization', 'network_in', 'request_count', 'storage_usage', 'estimated_cost']

Feature matrix shape:
(14400, 6)


In [12]:
model = IsolationForest(
    n_estimators=200,
    contamination=0.007,
    random_state=42,
    n_jobs=-1
)

model.fit(X)

print("Isolation Forest trained successfully!")

Isolation Forest trained successfully!


In [13]:
predictions = model.predict(X)

cloud_df["ml_prediction"] = np.where(
    predictions == -1,
    1,
    0
)

print("Predictions generated successfully!")

print(
    cloud_df["ml_prediction"].value_counts()
)

Predictions generated successfully!
ml_prediction
0    14299
1      101
Name: count, dtype: int64


In [14]:
comparison = pd.crosstab(
    cloud_df["is_anomaly"],
    cloud_df["ml_prediction"],
    rownames=["Actual"],
    colnames=["Predicted"]
)

comparison

Predicted,0,1
Actual,,
0,14242,58
1,57,43


In [15]:
y_true = cloud_df["is_anomaly"]
y_pred = cloud_df["ml_prediction"]

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Precision: 0.4257
Recall:    0.4300
F1 Score:  0.4279


In [16]:
detected_anomalies = cloud_df[
    cloud_df["ml_prediction"] == 1
].copy()

print(f"Number of anomalies detected: {len(detected_anomalies)}")

detected_anomalies[
    [
        "resource_id",
        "resource_type",
        "timestamp",
        "cpu_utilization",
        "memory_utilization",
        "network_in",
        "request_count",
        "estimated_cost",
        "is_anomaly",
        "anomaly_type"
    ]
].head(20)

Number of anomalies detected: 101


,resource_id,resource_type,timestamp,cpu_utilization,memory_utilization,network_in,request_count,estimated_cost,is_anomaly,anomaly_type
290,EC2-001,Web Server,2026-08-04 00:30:00,95.461754,24.541821,54.901605,724.621218,0.060710,1,cpu_spike
484,EC2-001,Web Server,2026-08-06 01:00:00,15.373090,25.577497,1198.207223,770.094557,0.057362,1,network_spike
690,EC2-001,Web Server,2026-08-08 04:30:00,16.801931,25.801721,1035.275986,744.586231,0.058077,1,network_spike
847,EC2-001,Web Server,2026-08-09 19:45:00,85.087447,86.676608,780.493286,4211.315071,0.219908,1,combined_spike
1270,EC2-001,Web Server,2026-08-14 05:30:00,22.431213,24.202865,59.067667,752.983898,0.237058,1,cost_spike
1879,EC2-001,Web Server,2026-08-20 13:45:00,85.702597,91.800887,862.638190,3803.394613,0.179284,1,combined_spike
2034,EC2-001,Web Server,2026-08-22 04:30:00,93.690694,30.682618,64.927061,829.358766,0.062299,1,cpu_spike
3860,EC2-002,Web Server,2026-08-11 05:00:00,94.034647,30.759237,58.999432,1001.721726,0.063768,1,cpu_spike
3952,EC2-002,Web Server,2026-08-12 04:00:00,96.124571,84.289675,723.376012,2208.313536,0.160374,1,combined_spike
5372,EC2-002,Web Server,2026-08-26 23:00:00,87.148333,85.391011,759.083437,3244.960170,0.193405,1,combined_spike


In [17]:
pd.crosstab(
    cloud_df["anomaly_type"],
    cloud_df["ml_prediction"],
    rownames=["Actual Anomaly Type"],
    colnames=["ML Prediction"]
)

ML Prediction,0,1
Actual Anomaly Type,,
combined_spike,0,15
cost_spike,15,5
cpu_spike,10,15
memory_spike,18,2
network_spike,14,6
normal,14242,58


In [18]:
feature_df = cloud_df.copy()

print(f"Feature engineering dataset shape: {feature_df.shape}")

Feature engineering dataset shape: (14400, 12)


In [19]:
feature_df["hour"] = feature_df["timestamp"].dt.hour

feature_df["day_of_week"] = feature_df["timestamp"].dt.dayofweek

feature_df["is_weekend"] = (
    feature_df["day_of_week"] >= 5
).astype(int)

print(
    feature_df[
        ["timestamp", "hour", "day_of_week", "is_weekend"]
    ].head()
)

            timestamp  hour  day_of_week  is_weekend
0 2026-08-01 00:00:00     0            5           1
1 2026-08-01 00:15:00     0            5           1
2 2026-08-01 00:30:00     0            5           1
3 2026-08-01 00:45:00     0            5           1
4 2026-08-01 01:00:00     1            5           1


In [20]:
feature_df["resource_utilization"] = (
    feature_df["cpu_utilization"]
    + feature_df["memory_utilization"]
    + feature_df["storage_usage"]
) / 3

feature_df["cpu_memory_ratio"] = (
    feature_df["cpu_utilization"]
    / (feature_df["memory_utilization"] + 1e-6)
)

print(
    feature_df[
        [
            "cpu_utilization",
            "memory_utilization",
            "storage_usage",
            "resource_utilization",
            "cpu_memory_ratio"
        ]
    ].head()
)

   cpu_utilization  memory_utilization  storage_usage  resource_utilization  \
0        19.087236           24.906175      54.324431             32.772614   
1        21.372462           28.249815      56.138281             35.253519   
2        20.178249           26.371394      54.555520             33.701721   
3        16.712111           24.016472      53.726745             31.485109   
4        19.658934           25.695524      54.572166             33.308874   

   cpu_memory_ratio  
0          0.766366  
1          0.756552  
2          0.765157  
3          0.695860  
4          0.765072  


In [21]:
feature_df["network_per_request"] = (
    feature_df["network_in"]
    / (feature_df["request_count"] + 1)
)

feature_df["requests_per_network"] = (
    feature_df["request_count"]
    / (feature_df["network_in"] + 1)
)

print(
    feature_df[
        [
            "network_in",
            "request_count",
            "network_per_request",
            "requests_per_network"
        ]
    ].head()
)

   network_in  request_count  network_per_request  requests_per_network
0   74.708176     899.351067             0.082977             11.879180
1   53.193879     739.607109             0.071825             13.647429
2   57.415241     773.460620             0.074136             13.240733
3   77.400133     922.063367             0.083851             11.760992
4   63.384328     790.523342             0.080079             12.278195


In [22]:
feature_df["cost_per_request"] = (
    feature_df["estimated_cost"]
    / (feature_df["request_count"] + 1)
)

feature_df["cost_per_cpu"] = (
    feature_df["estimated_cost"]
    / (feature_df["cpu_utilization"] + 1e-6)
)

print(
    feature_df[
        [
            "estimated_cost",
            "request_count",
            "cpu_utilization",
            "cost_per_request",
            "cost_per_cpu"
        ]
    ].head()
)

   estimated_cost  request_count  cpu_utilization  cost_per_request  \
0        0.059451     899.351067        19.087236          0.000066   
1        0.060514     739.607109        21.372462          0.000082   
2        0.059651     773.460620        20.178249          0.000077   
3        0.058352     922.063367        16.712111          0.000063   
4        0.059368     790.523342        19.658934          0.000075   

   cost_per_cpu  
0      0.003115  
1      0.002831  
2      0.002956  
3      0.003492  
4      0.003020  


In [23]:
feature_columns_v2 = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "storage_usage",
    "estimated_cost",
    "resource_utilization",
    "cpu_memory_ratio",
    "network_per_request",
    "requests_per_network",
    "cost_per_request",
    "cost_per_cpu",
    "hour",
    "day_of_week",
    "is_weekend"
]

X_v2 = feature_df[feature_columns_v2].copy()

print("Improved feature matrix:")
print(X_v2.shape)

print("\nFeatures:")
for feature in feature_columns_v2:
    print("-", feature)

Improved feature matrix:
(14400, 15)

Features:
- cpu_utilization
- memory_utilization
- network_in
- request_count
- storage_usage
- estimated_cost
- resource_utilization
- cpu_memory_ratio
- network_per_request
- requests_per_network
- cost_per_request
- cost_per_cpu
- hour
- day_of_week
- is_weekend


In [24]:
X_v2.describe()

,cpu_utilization,memory_utilization,network_in,request_count,storage_usage,estimated_cost,resource_utilization,cpu_memory_ratio,network_per_request,requests_per_network,cost_per_request,cost_per_cpu,hour,day_of_week,is_weekend
count,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000,14400.000000
mean,35.012860,47.652143,111.861720,1360.106784,59.003960,0.075064,47.222988,0.733811,0.088078,12.017781,0.000066,0.002317,11.500000,3.166667,0.333333
std,12.088002,14.856742,60.154671,643.239929,8.009445,0.013105,10.347633,0.116117,0.052215,2.500756,0.000030,0.000601,6.922427,2.034497,0.471421
min,8.806631,18.876900,22.122866,177.327826,44.573017,0.052340,25.958727,0.148343,0.037369,0.279929,0.000030,0.000581,0.000000,0.000000,0.000000
25%,25.429363,34.930664,77.982010,858.707425,54.292023,0.065559,39.585031,0.674821,0.072632,10.609960,0.000045,0.001921,5.750000,1.000000,0.000000
50%,33.977866,48.653194,103.675501,1268.230237,57.979967,0.073892,46.148333,0.733273,0.080346,12.318145,0.000061,0.002178,11.500000,3.000000,0.000000
75%,43.905551,58.138590,135.635156,1753.860450,62.682345,0.084796,54.077522,0.788836,0.093053,13.638863,0.000080,0.002585,17.250000,5.000000,1.000000
max,98.969779,97.811569,1198.207223,4598.935313,75.451858,0.336731,87.334834,3.889758,3.552374,25.604936,0.000617,0.013980,23.000000,6.000000,1.000000


In [25]:
feature_columns_v2 = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "storage_usage",
    "estimated_cost",
    "resource_utilization",
    "cpu_memory_ratio",
    "network_per_request",
    "requests_per_network",
    "cost_per_request",
    "cost_per_cpu"
]

X_v2 = feature_df[feature_columns_v2].copy()

print("V2 feature matrix shape:", X_v2.shape)

V2 feature matrix shape: (14400, 12)


In [26]:
model_v2 = IsolationForest(
    n_estimators=200,
    contamination=0.007,
    random_state=42,
    n_jobs=-1
)

model_v2.fit(X_v2)

print("Isolation Forest V2 trained successfully!")

Isolation Forest V2 trained successfully!


In [27]:
predictions_v2 = model_v2.predict(X_v2)

feature_df["ml_prediction_v2"] = np.where(
    predictions_v2 == -1,
    1,
    0
)

print("V2 predictions generated successfully!")

print(
    feature_df["ml_prediction_v2"].value_counts()
)

V2 predictions generated successfully!
ml_prediction_v2
0    14299
1      101
Name: count, dtype: int64


In [28]:
y_true = feature_df["is_anomaly"]
y_pred_v2 = feature_df["ml_prediction_v2"]

precision_v2 = precision_score(y_true, y_pred_v2)
recall_v2 = recall_score(y_true, y_pred_v2)
f1_v2 = f1_score(y_true, y_pred_v2)

print(f"V2 Precision: {precision_v2:.4f}")
print(f"V2 Recall:    {recall_v2:.4f}")
print(f"V2 F1 Score:  {f1_v2:.4f}")

V2 Precision: 0.4257
V2 Recall:    0.4300
V2 F1 Score:  0.4279


In [29]:
same_predictions = np.array_equal(
    cloud_df["ml_prediction"].values,
    feature_df["ml_prediction_v2"].values
)

print("Are baseline and V2 predictions identical?", same_predictions)

Are baseline and V2 predictions identical? False


In [30]:
pd.crosstab(
    cloud_df["anomaly_type"],
    feature_df["ml_prediction_v2"],
    rownames=["Actual Anomaly Type"],
    colnames=["V2 Prediction"]
)

V2 Prediction,0,1
Actual Anomaly Type,,
combined_spike,0,15
cost_spike,10,10
cpu_spike,15,10
memory_spike,20,0
network_spike,12,8
normal,14242,58


In [31]:
# Save the V2 dataset with ML predictions

V2_DATA_FILE = PROCESSED_DATA_DIR / "cloudsentinel_v2_predictions.csv"

feature_df.to_csv(
    V2_DATA_FILE,
    index=False
)

print(f"V2 dataset saved to: {V2_DATA_FILE}")

V2 dataset saved to: c:\Users\Abhishek\Desktop\CloudSentinel-AI\data\processed\cloudsentinel_v2_predictions.csv
